# EvoRank foundation (Phase 0 / Phase 1)

This notebook de-risks the foundation **before** you spend any LLM tokens on the SkyDiscover loop. It runs end to end on synthetic data right now, and has a clearly marked cell to swap in the real Expedia data later.

What it validates:
1. The three query-grouped metrics (NDCG, book-NDCG, revenue) on a hand example.
2. A baseline XGBoost ranker with built-in `rank:ndcg`.
3. The group-aware LambdaMART custom objective (the bug-prone piece), checked against the built-in objective.
4. The SkyDiscover `evaluate(program_path)` contract, including the degeneracy rejection guard.
5. One manual evolution step, so you can see exactly what the agent will be doing.

Port the verified functions from here into `metrics.py`, `train_xgbranker.py`, `seed/initial_program.py`, and `eval/evaluator.py`. Keep this notebook for interactive debugging.

In [1]:
import importlib.util
import numpy as np
import pandas as pd
import xgboost as xgb

SEED = 0
np.random.seed(SEED)
print("xgboost", xgb.__version__)

xgboost 3.3.0


## A. Data

A synthetic dataset that mirrors Expedia's structure: query groups, a graded label in `{0, 1, 5}` (5 = booked, 1 = clicked, 0 = neither), a `booking` flag, a `price`, and a `rev` column (`price` when booked, else 0). Some queries get a second booking so the revenue objective genuinely differs from the booking objective.

To use the real data instead, replace `make_synthetic(...)` with the Expedia loader in the commented cell below. Everything downstream is identical.

In [2]:
def make_synthetic(n_queries=500, items=15, n_feat=8, seed=0):
    rng = np.random.default_rng(seed)
    rows = []
    for q in range(n_queries):
        m = int(rng.integers(items - 4, items + 4))
        X = rng.normal(size=(m, n_feat))
        w = rng.normal(size=n_feat)
        utility = X @ w + rng.normal(scale=0.5, size=m)
        order = np.argsort(-utility)
        rel = np.zeros(m, dtype=int)
        # best item: booked (5) or clicked (1)
        rel[order[0]] = 5 if rng.random() < 0.6 else 1
        # second item: sometimes a second booking, so revenue != booking
        if m > 1 and rng.random() < 0.2:
            rel[order[1]] = 5
        # a few mid items clicked
        for idx in order[2:max(2, m // 4)]:
            rel[idx] = 1 if rng.random() < 0.5 else 0
        price = rng.uniform(80, 600, size=m)
        booking = (rel == 5).astype(int)
        rev = booking * price
        for i in range(m):
            rows.append((q, *X[i], int(rel[i]), int(booking[i]), float(price[i]), float(rev[i])))
    cols = ["qid"] + [f"f{i}" for i in range(n_feat)] + ["rel", "booking", "price", "rev"]
    return pd.DataFrame(rows, columns=cols)

df = make_synthetic(seed=1)
print(df.shape)
df.head()

(7217, 13)


,qid,f0,f1,f2,f3,f4,f5,f6,f7,rel,booking,price,rev
0,0,0.821618,0.330437,-1.303157,0.905356,0.446375,-0.536953,0.581118,0.364572,0,0,340.185344,0.0
1,0,0.294132,0.028422,0.546713,-0.736454,-0.162910,-0.482119,0.598846,0.039722,0,0,120.083580,0.0
2,0,-0.292457,-0.781908,-0.257192,0.008142,-0.275603,1.294064,1.006724,-2.711162,0,0,333.993598,0.0
3,0,-1.889013,-0.174772,-0.422190,0.213643,0.217322,2.117839,-1.112021,-0.377605,0,0,190.672118,0.0
4,0,2.042772,0.646703,0.663063,-0.514006,-1.648075,0.167465,0.109014,-1.227352,0,0,149.002075,0.0


### Swap in real Expedia (run later, once `train.csv` is downloaded)

```python
# raw = pd.read_csv("data/raw/train.csv")
# raw["rel"] = np.where(raw["booking_bool"] == 1, 5,
#              np.where(raw["click_bool"] == 1, 1, 0))
# raw["booking"] = raw["booking_bool"]
# raw["rev"] = raw["gross_bookings_usd"].fillna(0.0)
# raw = raw.rename(columns={"srch_id": "qid"})
# feature_cols = [...]              # numeric search/hotel/comp columns; drop position, *_bool, gross_bookings_usd
# raw = raw.rename(columns={c: f"f{i}" for i, c in enumerate(feature_cols)})
# df = raw[["qid"] + [f"f{i}" for i in range(len(feature_cols))] + ["rel", "booking", "price", "rev"]]
```
The only contract downstream code needs: a `qid` column, feature columns named `f0..fK`, and `rel` / `booking` / `rev` label columns. Sort by `qid` so groups are contiguous.

In [3]:
FEATS = [c for c in df.columns if c.startswith("f")]

def split_by_qid(df, fracs=(0.70, 0.15), seed=7):
    qids = df["qid"].unique().copy()
    rng = np.random.default_rng(seed)
    rng.shuffle(qids)
    n = len(qids)
    tr_q = set(qids[: int(fracs[0] * n)])
    va_q = set(qids[int(fracs[0] * n): int((fracs[0] + fracs[1]) * n)])
    te_q = set(qids) - tr_q - va_q
    pick = lambda s: df[df.qid.isin(s)].sort_values("qid").reset_index(drop=True)
    return pick(tr_q), pick(va_q), pick(te_q)

train_df, val_df, test_df = split_by_qid(df)
# no query may span folds
assert not (set(train_df.qid) & set(val_df.qid) & set(test_df.qid))
print(f"queries  train {train_df.qid.nunique()}  val {val_df.qid.nunique()}  test {test_df.qid.nunique()}")
print(f"rows     train {len(train_df)}  val {len(val_df)}  test {len(test_df)}")

queries  train 350  val 75  test 75
rows     train 5071  val 1085  test 1061


## B. Metrics

All three are NDCG-style, computed per query group and averaged over queries. They differ only in the gain:
- `ndcg` uses graded relevance gain `2**rel - 1`.
- `book_ndcg` uses the binary `booking` flag.
- `revenue` uses `rev` (booking-weighted price), an explicit revenue **proxy**.

In [4]:
def _group_slices(group_sizes):
    idx, out = 0, []
    for g in group_sizes:
        out.append((idx, idx + int(g))); idx += int(g)
    return out

def grouped_ndcg(gain, scores, group_sizes, k=10):
    vals = []
    for s, e in _group_slices(group_sizes):
        g, sc = gain[s:e], scores[s:e]
        order = np.argsort(-sc, kind="stable")
        ranked = g[order][:k]
        dcg = float(np.sum(ranked * (1.0 / np.log2(np.arange(2, len(ranked) + 2)))))
        ideal = np.sort(g)[::-1][:k]
        idcg = float(np.sum(ideal * (1.0 / np.log2(np.arange(2, len(ideal) + 2)))))
        if idcg > 0:
            vals.append(dcg / idcg)
    return float(np.mean(vals)) if vals else 0.0

def group_sizes_of(frame):
    return frame.groupby("qid", sort=False).size().to_numpy()

def metrics_bundle(frame, scores, k=10):
    gs = group_sizes_of(frame)
    return {
        "ndcg":      grouped_ndcg((2.0 ** frame["rel"].to_numpy() - 1.0), scores, gs, k),
        "book_ndcg": grouped_ndcg(frame["booking"].to_numpy().astype(float), scores, gs, k),
        "revenue":   grouped_ndcg(frame["rev"].to_numpy().astype(float), scores, gs, k),
    }

# unit check: perfect ranking scores 1.0, reversed scores below 1.0
_g = np.array([3., 2., 3., 0., 1., 2.])
assert abs(grouped_ndcg(_g, _g, [6], 6) - 1.0) < 1e-9
assert grouped_ndcg(_g, -_g, [6], 6) < 1.0
print("metric unit checks passed")

metric unit checks passed


## C. Baseline ranker (built-in `rank:ndcg`)

A plain XGBoost ranker. This is both a sanity baseline and the reference the custom objective must match in the next section.

In [5]:
def to_dmatrix(frame):
    d = xgb.DMatrix(frame[FEATS].to_numpy(), label=frame["rel"].to_numpy().astype(float))
    d.set_group(group_sizes_of(frame))
    return d

FIXED_PARAMS = {"eta": 0.1, "max_depth": 6, "min_child_weight": 0.1,
                "tree_method": "hist", "seed": SEED}
FIXED_ROUNDS = 120

def train_builtin(train_df, val_df):
    bst = xgb.train({**FIXED_PARAMS, "objective": "rank:ndcg"},
                    to_dmatrix(train_df), num_boost_round=FIXED_ROUNDS)
    return bst.predict(to_dmatrix(val_df))

rand_scores = np.random.default_rng(0).normal(size=len(val_df))
print("random   ", {k: round(v, 4) for k, v in metrics_bundle(val_df, rand_scores).items()})
builtin_pred = train_builtin(train_df, val_df)
builtin_m = metrics_bundle(val_df, builtin_pred)
print("built-in ", {k: round(v, 4) for k, v in builtin_m.items()})

random    {'ndcg': 0.3538, 'book_ndcg': 0.3217, 'revenue': 0.3185}


built-in  {'ndcg': 0.4007, 'book_ndcg': 0.3662, 'revenue': 0.3628}


## D. Group-aware LambdaMART objective

This is the piece most likely to hide a bug, so it gets validated directly. The function computes, per query, the pairwise lambda-gradient with `|deltaNDCG|` weighting (Burges' LambdaMART), and returns `(grad, hess)` per row.

Two checks:
- `use_ndcg_weight=True` should reproduce the built-in `rank:ndcg` numbers within noise. That is the correctness check.
- `use_ndcg_weight=False` (plain RankNet, no metric weighting) should be measurably worse, which confirms the `deltaNDCG` term is wired correctly.

The signature `lambdamart_objective(predt, labels, group_slices, ...)` is exactly what goes in the seed program's EVOLVE-BLOCK.

In [6]:
def lambdamart_objective(predt, labels, group_slices, sigma=1.0, use_ndcg_weight=True):
    grad = np.zeros_like(predt, dtype=np.float64)
    hess = np.zeros_like(predt, dtype=np.float64)
    for s, e in group_slices:
        sc = predt[s:e].astype(np.float64); lab = labels[s:e]; m = len(sc)
        if m < 2:
            continue
        S = sc[:, None] - sc[None, :]
        rho = 1.0 / (1.0 + np.exp(sigma * S))          # sigmoid(-sigma * S)
        M = (lab[:, None] > lab[None, :])              # i strictly more relevant than j
        if use_ndcg_weight:
            gain = (2.0 ** lab - 1.0)
            ranks = np.empty(m, dtype=int)
            ranks[np.argsort(-sc, kind="stable")] = np.arange(m)
            disc = 1.0 / np.log2(ranks + 2.0)
            idcg = float(np.sum(np.sort(gain)[::-1] * (1.0 / np.log2(np.arange(2, m + 2))))) or 1.0
            dZ = np.abs((gain[:, None] - gain[None, :]) * (disc[:, None] - disc[None, :])) / idcg
        else:
            dZ = np.ones((m, m))
        lam = np.where(M, -sigma * rho * dZ, 0.0)
        hmat = np.where(M, (sigma ** 2) * rho * (1.0 - rho) * dZ, 0.0)
        grad[s:e] = lam.sum(axis=1) - lam.sum(axis=0)  # i gets lam, j gets -lam
        hess[s:e] = hmat.sum(axis=1) + hmat.sum(axis=0)
    return grad, np.maximum(hess, 1e-6)                # keep hessian positive

def make_xgb_obj(group_slices, **kw):
    def obj(predt, dtrain):
        g, h = lambdamart_objective(predt, dtrain.get_label(), group_slices, **kw)
        if not (np.all(np.isfinite(g)) and np.all(np.isfinite(h))):
            raise ValueError("non-finite grad/hess")
        return g, h
    return obj

def train_custom(train_df, val_df, **kw):
    dtr = to_dmatrix(train_df)
    slices = _group_slices(group_sizes_of(train_df))
    bst = xgb.train({**FIXED_PARAMS, "base_score": 0.0, "disable_default_eval_metric": 1},
                    dtr, num_boost_round=FIXED_ROUNDS, obj=make_xgb_obj(slices, **kw))
    return bst.predict(to_dmatrix(val_df))

custom_m  = metrics_bundle(val_df, train_custom(train_df, val_df, use_ndcg_weight=True))
ranknet_m = metrics_bundle(val_df, train_custom(train_df, val_df, use_ndcg_weight=False))
print("built-in       ", {k: round(v, 4) for k, v in builtin_m.items()})
print("custom NDCG    ", {k: round(v, 4) for k, v in custom_m.items()})
print("custom RankNet ", {k: round(v, 4) for k, v in ranknet_m.items()})

# correctness checks
assert custom_m["ndcg"] > builtin_m["ndcg"] - 0.03, "custom objective should track built-in rank:ndcg"
assert custom_m["ndcg"] > ranknet_m["ndcg"], "NDCG weighting should beat plain RankNet"
print("\ncustom objective validated against built-in rank:ndcg")

built-in        {'ndcg': 0.4007, 'book_ndcg': 0.3662, 'revenue': 0.3628}
custom NDCG     {'ndcg': 0.4071, 'book_ndcg': 0.3913, 'revenue': 0.3812}
custom RankNet  {'ndcg': 0.3399, 'book_ndcg': 0.3343, 'revenue': 0.3267}

custom objective validated against built-in rank:ndcg


## E. The seed program and the `evaluate(program_path)` contract

SkyDiscover passes a path to a candidate program and calls `evaluate(program_path)`. Here we:
1. Write the seed program to a file, with the objective wrapped in `EVOLVE-BLOCK` markers (in the repo this lives at `../seed/initial_program.py`).
2. Define `evaluate(program_path)` exactly as the SkyDiscover evaluator will: import the candidate, train with its objective, return the three metrics plus `combined_score` and an `artifacts.feedback` string.
3. Confirm the rejection guard catches a degenerate objective rather than letting it score.

In [7]:
SEED_SRC = '''import numpy as np

# EVOLVE-BLOCK-START: objective
def lambdamart_objective(predt, labels, group_slices, sigma=1.0):
    """Group-aware LambdaMART gradient. SkyDiscover mutates this body."""
    grad = np.zeros_like(predt, dtype=np.float64)
    hess = np.zeros_like(predt, dtype=np.float64)
    for s, e in group_slices:
        sc = predt[s:e].astype(np.float64); lab = labels[s:e]; m = len(sc)
        if m < 2:
            continue
        S = sc[:, None] - sc[None, :]
        rho = 1.0 / (1.0 + np.exp(sigma * S))
        M = (lab[:, None] > lab[None, :])
        gain = (2.0 ** lab - 1.0)
        ranks = np.empty(m, dtype=int); ranks[np.argsort(-sc, kind="stable")] = np.arange(m)
        disc = 1.0 / np.log2(ranks + 2.0)
        idcg = float(np.sum(np.sort(gain)[::-1] * (1.0 / np.log2(np.arange(2, m + 2))))) or 1.0
        dZ = np.abs((gain[:, None] - gain[None, :]) * (disc[:, None] - disc[None, :])) / idcg
        lam = np.where(M, -sigma * rho * dZ, 0.0)
        hmat = np.where(M, (sigma ** 2) * rho * (1.0 - rho) * dZ, 0.0)
        grad[s:e] = lam.sum(axis=1) - lam.sum(axis=0)
        hess[s:e] = hmat.sum(axis=1) + hmat.sum(axis=0)
    return grad, np.maximum(hess, 1e-6)
# EVOLVE-BLOCK-END: objective
'''

with open("initial_program_seed.py", "w") as f:
    f.write(SEED_SRC)

def load_program(path):
    spec = importlib.util.spec_from_file_location("candidate", path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

def one_line_diagnostic(m):
    return f"ndcg={m['ndcg']:.4f} book={m['book_ndcg']:.4f} rev={m['revenue']:.4f}"

def evaluate(program_path):
    try:
        mod = load_program(program_path)
        dtr = to_dmatrix(train_df)
        slices = _group_slices(group_sizes_of(train_df))
        def obj(predt, dtrain):
            g, h = mod.lambdamart_objective(predt, dtrain.get_label(), slices)
            if not (np.all(np.isfinite(g)) and np.all(np.isfinite(h))):
                raise ValueError("non-finite grad/hess")
            return g, h
        bst = xgb.train({**FIXED_PARAMS, "base_score": 0.0, "disable_default_eval_metric": 1},
                        dtr, num_boost_round=FIXED_ROUNDS, obj=obj)
        m = metrics_bundle(val_df, bst.predict(to_dmatrix(val_df)))
        return {**m, "combined_score": m["ndcg"], "artifacts": {"feedback": one_line_diagnostic(m)}}
    except Exception as ex:
        return {"ndcg": 0.0, "book_ndcg": 0.0, "revenue": 0.0, "combined_score": 0.0,
                "artifacts": {"feedback": f"rejected: {type(ex).__name__}: {ex}"}}

print("seed program :", evaluate("initial_program_seed.py"))

# degenerate candidate (NaN gradient) must be rejected, not scored
with open("broken_seed.py", "w") as f:
    f.write("import numpy as np\n"
            "def lambdamart_objective(predt, labels, group_slices, sigma=1.0):\n"
            "    return np.full_like(predt, np.nan), np.ones_like(predt)\n")
print("broken program:", evaluate("broken_seed.py"))

seed program : {'ndcg': 0.40707047972001265, 'book_ndcg': 0.39134542258786353, 'revenue': 0.3811826080791582, 'combined_score': 0.40707047972001265, 'artifacts': {'feedback': 'ndcg=0.4071 book=0.3913 rev=0.3812'}}
broken program: {'ndcg': 0.0, 'book_ndcg': 0.0, 'revenue': 0.0, 'combined_score': 0.0, 'artifacts': {'feedback': 'rejected: ValueError: non-finite grad/hess'}}


## F. One manual evolution step

This is exactly what SkyDiscover automates: take the seed objective, change the body, re-evaluate, compare. Here we hand-write a "mutated" candidate that drops the `deltaNDCG` weighting (a deliberately worse edit) and confirm the evaluator scores it lower. When you run the real loop, the LLM proposes these edits and the Pareto database keeps the ones that improve the frontier.

In [8]:
MUTANT_SRC = '''import numpy as np

# EVOLVE-BLOCK-START: objective
def lambdamart_objective(predt, labels, group_slices, sigma=1.0):
    # mutation: plain RankNet, no deltaNDCG weighting (expected to be worse)
    grad = np.zeros_like(predt, dtype=np.float64)
    hess = np.zeros_like(predt, dtype=np.float64)
    for s, e in group_slices:
        sc = predt[s:e].astype(np.float64); lab = labels[s:e]; m = len(sc)
        if m < 2:
            continue
        S = sc[:, None] - sc[None, :]
        rho = 1.0 / (1.0 + np.exp(sigma * S))
        M = (lab[:, None] > lab[None, :])
        lam = np.where(M, -sigma * rho, 0.0)
        hmat = np.where(M, (sigma ** 2) * rho * (1.0 - rho), 0.0)
        grad[s:e] = lam.sum(axis=1) - lam.sum(axis=0)
        hess[s:e] = hmat.sum(axis=1) + hmat.sum(axis=0)
    return grad, np.maximum(hess, 1e-6)
# EVOLVE-BLOCK-END: objective
'''
with open("mutant_seed.py", "w") as f:
    f.write(MUTANT_SRC)

base = evaluate("initial_program_seed.py")
mut  = evaluate("mutant_seed.py")
print("seed   :", base["artifacts"]["feedback"])
print("mutant :", mut["artifacts"]["feedback"])
print(f"\ndelta ndcg (mutant - seed): {mut['ndcg'] - base['ndcg']:+.4f}  (negative = mutation was worse, as expected)")

seed   : ndcg=0.4071 book=0.3913 rev=0.3812
mutant : ndcg=0.3399 book=0.3343 rev=0.3267

delta ndcg (mutant - seed): -0.0672  (negative = mutation was worse, as expected)


## G. Next steps

1. Run the Expedia loader cell in Section A on the real `train.csv`, rerun this notebook, and confirm the correctness checks still pass on real data. Subsample to roughly 25 to 40k queries so each `evaluate` call stays under about 60 seconds.
2. Port the verified functions out of this notebook:
   - metrics to `ltr/metrics.py`
   - `to_dmatrix` and the training harness to `ltr/train_xgbranker.py`
   - the objective (the `EVOLVE-BLOCK` body) to `seed/initial_program.py`
   - `evaluate(program_path)` to `eval/evaluator.py`
3. Then follow the CLAUDE.md build order: baselines (tuned LambdaMART, LambdaLoss, random search at equal budget), then the two evolutionary configs, then the memory ablation.

Do not start the evolutionary runs until this notebook's checks pass on the real data. The custom objective matching built-in `rank:ndcg` is the signal that the foundation is sound.